# DSPy Prompt Optimisation: Healthcare Domain Classifier

**Question:** Can DSPy recover enough accuracy to make `gpt-4.1-mini` viable against the full `gpt-5.4` production prompt?

**Reading order:** setup → classifier and metric → data split → baselines → optimisers → held-out results. The long original prompt is retained as an optional reference; it is not executed.

## Experiment at a glance

The classifier routes one healthcare utterance to a **primary domain** and **sub-domain**. The taxonomy has five primary domains: `AppointmentManagement`, `PaymentBillingSupport`, `MedicationPrescriptionAssistance`, `HealthRecordsAccess`, and `GeneralHealthInquiry`. Fine-grained distinctions—including sensitive-record exclusions—make sub-domain accuracy the hard part.

The reference program uses `gpt-5.4` and the complete original prompt content. The candidate program uses `gpt-4.1-mini` with the same taxonomy and is optimised two ways:

- **Bootstrap few-shot search:** selects demonstrations from labelled training examples.
- **GEPA:** uses feedback on failures to evolve the instruction.

All optimisers see only train/validation data. The 80-example test set is used once for final comparison. Exact match requires both labels; a correct primary domain with the wrong sub-domain earns partial credit.

## 1. Setup

Install the listed packages once, make sure `OPENAI_API_KEY` is available through the environment or `.env`, then run this section. Caching is disabled so timing reflects real API calls.

In [ ]:
# Run once if needed:
# %pip install dspy pandas openpyxl scikit-learn python-dotenv

In [ ]:
from pathlib import Path
import dspy
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

load_dotenv(override=True)  # Prefer the project .env to stale notebook-session values.

SEED = 0
STUDENT_MODEL   = "openai/gpt-4.1-mini"
TEACHER_MODEL   = "openai/gpt-4o-mini"
REFERENCE_MODEL = "openai/gpt-5.4"
MAX_TOKENS = 512

lm = dspy.LM(model=STUDENT_MODEL, max_tokens=MAX_TOKENS, cache=False)
dspy.configure(lm=lm)
print(f"Configured student: {STUDENT_MODEL}")

## 2. DSPy classifier and scoring

The production system prompt encodes the full domain taxonomy, its disambiguation rules, and six hand-authored examples. The reference baseline below uses all of that content with `gpt-5.4`.

For DSPy we adapt it to **single-utterance classification**. The production prompt is designed to classify a list of utterances in one call (to amortise context overhead); here we simplify to one utterance at a time so that DSPy can:
1. Reason about individual examples when selecting demonstrations
2. Score each training example independently against the metric
3. Inject retrieved few-shot examples cleanly into a structured prompt

DSPy scores one item at a time, so the evaluation harness uses a single-utterance representation. It preserves every original classification rule and every example utterance/label pair, while changing only the batch JSON transport contract.

> **Important:** The reference is the complete production prompt content on `gpt-5.4`; the haiku zero-shot configuration appears later as the student baseline.

In [ ]:
STUDENT_INSTRUCTIONS = """\
You are a highly specialized AI assistant and an expert domain classifier in the healthcare domain.
You are given a single utterance from a user query in the healthcare domain.
Your task is to classify it into one of the supported primary Domains and sub-domains listed below.

**Instructions:**
1.  Analyze each specific utterance provided in the input list in the context of the conversation history (if any).
2.  Assign the most appropriate primary domain and sub-domain from the supported list below. A valid classification is mandatory and cannot be null.

**Critical Constraints:**
- Return exactly one classification for the supplied utterance.
- Select a valid `sub_domain` under the assigned `domain`.
- Distinguish informational queries (usually `GeneralHealthInquiry`) from transactional HealthHub e-service requests. A request that maps to a transactional domain remains transactional regardless of phrasing.
- Treat technical e-service errors as `Issues` under the applicable transactional domain when that label is available in the deployed taxonomy.

**Supported Primary Domains and Sub-Domains:**
-   `AppointmentManagement`: For queries related to scheduling, rescheduling, managing medical appointments, or appointment registration, applicable to both the user and their dependents.
      -   `View`: Queries about viewing or checking existing appointment details.
      -   `Schedule`: Queries about booking or scheduling a new appointment or requesting for available appointment slots.
      -   `Reschedule`: Queries about rescheduling or changing the date or time of an existing appointment.
      -   `Cancel`: Queries about cancellation of an existing appointment.
      -   `Register`: Queries about pre-registration, obtaining of queue numbers and checking queue status for appointments.
      -   `GeneralInquiry`: General queries about appointments, including preparation, scheduling advice, doctor/consultation type changes, or administrative processes (e.g., referrals, subsidy eligibility).

-   `PaymentBillingSupport`: For queries related to billing or payments, applicable to both the user and their dependents.
      -   `ViewOutstandingBalance`: Queries about checking or viewing current outstanding medical bills or charges.
      -   `RequestInvoice`: Queries about checking or viewing medical bills or invoices.
      -   `MakePayment`: Queries about making payments for medical bills.
      -   `GeneralInquiry`: General queries about bill payments, including understanding bill statuses, payment methods, itemized charges, payment reversals, fees, processing timelines, or billing disputes.

-   `MedicationPrescriptionAssistance`: For queries related to medication assistance, prescriptions, or medication management, applicable to both the user and their dependents.
      -   `RefillPrescription`: Queries about refilling prescriptions or medications and viewing medication balance.
      -   `RenewPrescription`: Queries about renewing or extending prescriptions or getting doctor approval to continue with prescribed medicines.
      -   `ViewRefillHistory`: Queries about viewing previously submitted medication refill requests or checking the status of medication refill requests (Note: This is not the same as prescription records).
      -   `GeneralInquiry`: General queries about medication refills, including managing submitted requests (e.g., editing, cancelling, collection points), delivery methods and fees, processing timelines, proxy collection, or cross-institution collection.

-   `HealthRecordsAccess`: For queries related to viewing, downloading, or requesting access to personal or dependent health records available via HealthHub (diagnostic reports, discharge summaries, screening results, allergy alerts, immunizations, medical certificates).
    **CRITICAL EXCLUSION (Sensitive Health Information):** You must NOT classify queries related to the following sensitive topics under `HealthRecordsAccess` as they are not displayed: HIV/STDs, Mental Health disorders (e.g., Schizophrenia), Substance Abuse, Termination of Pregnancy, Attempted Suicide, or Sexual Assault. Classify these as `GeneralHealthcareServicesRelated`.
      -   `AccessImmunizationRecords`: Queries about accessing vaccination records (NIR/NEHR) including NCIS/NAIS vaccines (e.g., Influenza, Tetanus, Hep B, Measles, Varicella/Chickenpox, HPV).
      -   `AccessHealthScreeningRecords`: Queries about accessing Healthier SG screening results (Cardiovascular, Cervical, Colorectal, and Breast Cancer).
      -   `AccessDischargeSummary`: Queries about accessing admission details, diagnosis, procedures, and prescriptions from Inpatient, Day Surgery, or A&E visits.
      -   `AccessDrugAllergyRecords`: Queries about accessing drug allergies, Adverse Drug Reactions (ADR), or G6PD status. (Note: Excludes food allergies).
      -   `AccessLabReports`: Queries about accessing general lab test reports (e.g., full blood count, lipid profile, liver function, urinalysis).
      -   `AccessRadiologyReports`: Queries about accessing radiology scans (e.g., X-ray, MRI, CT scan, Mammogram, Ultrasound).
      -   `AccessHistologyCytologyReports`: Queries about accessing biopsy results, Pap smears, or Fine Needle Aspiration Cytology (FNAC).
      -   `AccessGeneticReports`: Queries specifically about accessing genetic tests for Familial Hypercholesterolaemia (FH).
      -   `GeneralInquiry`: General queries about health records, including the timeline for health records to appear on HealthHub, processing delays, urgent record requests, or updating personal health details (drug allergies, adverse reactions, immunizations)

-   `GeneralHealthInquiry`: For general health-related questions, medical information, symptoms, wellness advice, or non-transactional health topics.
      -   `MedicationRelated`: Any informational query about medications, drug information, side effects, or usage (e.g., \"What are the side effects of Metformin?\", \"Can I take aspirin with ibuprofen?\").
      -   `MentalHealthSupport`: Any informational query about mental health, counseling, therapy, or emotional well-being (e.g., \"I feel very anxious and stressed lately.\", \"How do I deal with loneliness?\").
      -   `GeneralHealthcareServicesRelated`: Any informational query about healthcare services and care pathways (what service is for, eligibility, referrals, children's health and school records, MRPP & MC, Healthier SG) without requiring to perform a transaction in HealthHub. Excluding queries about the 4 transactional e-service domains listed above.
      -   `GeneralHealthRelated`: Any other general health-related informational query not covered by the above sub-domains.
      -   `HealthcareInstitutionInformation`: Queries about healthcare institution locations, directions, contact information, organizational affiliation, comparisons between healthcare providers, recommendations for treatments or services, or healthcare facility selection guidance (e.g., \"Where can I go for blood test?\", \"What's the phone number for HealthHub?\", \"Which hospital is better for cardiac surgery?\").
      -   `Emergency`: Urgent medical symptoms or emergencies requiring immediate medical attention (e.g., \"I'm having chest pain and difficulty breathing\", \"I think I'm having a heart attack\"), including symptoms that are borderline between side effects and emergencies.
"""

# Full production instructions add the 6 hand-authored examples from the original prompt.
# Used only for the production reference run — the student starts without any examples
# so Bootstrap can inject its own from a clean slate.
PROD_INSTRUCTIONS = STUDENT_INSTRUCTIONS + """\

## Examples

Example 1:
Utterances: [\"What are the side effects of Panadol?\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"What are the side effects of Panadol?\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"MedicationRelated\"}]}

Example 2:
Utterances: [\"I need to book a GP appointment for tomorrow.\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"I need to book a GP appointment for tomorrow.\", \"domain\": \"AppointmentManagement\", \"sub_domain\": \"Schedule\"}]}

Example 3 (Strict Adherence to 1-to-1 Mapping):
Utterances: [\"I am feeling depressed, what can I do?\", \"Can you help me formulate a exercise plan to feel better?\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"I am feeling depressed, what can I do?\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"MentalHealthSupport\"}, {\"utterance\": \"Can you help me formulate a exercise plan to feel better?\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"GeneralHealthRelated\"}]}

Example 4 (Mixed Intent/Domain):
Utterances: [\"I feel really hopeless and sad.\", \"Can you also help me pay my bill?\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"I feel really hopeless and sad.\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"MentalHealthSupport\"}, {\"utterance\": \"Can you also help me pay my bill?\", \"domain\": \"PaymentBillingSupport\", \"sub_domain\": \"MakePayment\"}]}

Example 5 (Distinction between Informational vs Transactional):
Utterances: [\"How do I refill my high blood pressure medicine?\", \"What happens if I stop taking it?\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"How do I refill my high blood pressure medicine?\", \"domain\": \"MedicationPrescriptionAssistance\", \"sub_domain\": \"RefillPrescription\"}, {\"utterance\": \"What happens if I stop taking it?\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"MedicationRelated\"}]}

Example 6 (Health Records & Sensitive Info Exclusion):
Utterances: [\"Can I download my X-ray results?\", \"Where is my HIV test result?\", \"I need a copy of my discharge summary.\"]
JSON Response: {\"classified_utterances\": [{\"utterance\": \"Can I download my X-ray results?\", \"domain\": \"HealthRecordsAccess\", \"sub_domain\": \"AccessRadiologyReports\"}, {\"utterance\": \"Where is my HIV test result?\", \"domain\": \"GeneralHealthInquiry\", \"sub_domain\": \"GeneralHealthcareServicesRelated\"}, {\"utterance\": \"I need a copy of my discharge summary.\", \"domain\": \"HealthRecordsAccess\", \"sub_domain\": \"AccessDischargeSummary\"}]}
"""


class DomainClassifierSignature(dspy.Signature):
    """Placeholder — replaced with instructions at runtime."""
    utterance: str  = dspy.InputField(desc="The user utterance to classify")
    domain: str     = dspy.OutputField(desc="Primary domain (e.g. AppointmentManagement)")
    sub_domain: str = dspy.OutputField(desc="Sub-domain (e.g. Schedule)")


StudentSignature = DomainClassifierSignature.with_instructions(STUDENT_INSTRUCTIONS)
ProdSignature    = DomainClassifierSignature.with_instructions(PROD_INSTRUCTIONS)


class DCModule(dspy.Module):
    def __init__(self, signature=None):
        self.predict = dspy.Predict(signature or StudentSignature)

    def forward(self, utterance: str) -> dspy.Prediction:
        return self.predict(utterance=utterance)

In [ ]:
def dc_metric(example, pred, trace=None) -> float:
    """1.0 = exact match, 0.5 = correct domain only, 0.0 = wrong."""
    pred_domain = (pred.domain or "").strip()
    pred_sub    = (pred.sub_domain or "").strip()
    if pred_domain == example.domain and pred_sub == example.sub_domain:
        return 1.0
    if (example.get("alternative_domain")
            and pred_domain == example.alternative_domain
            and pred_sub == example.alternative_sub_domain):
        return 1.0
    if pred_domain == example.domain:
        return 0.5
    return 0.0


def dc_metric_strict(example, pred, trace=None) -> bool:
    """Binary version used for Bootstrap demo selection."""
    return dc_metric(example, pred) == 1.0

## 3. Dataset and split

The dataset contains **321 real healthcare chatbot utterances** after filtering to records with at least one information-retrieval sub-query. Splits are stratified by primary domain.

We split the data as follows:

| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 191 | Source of demonstrations for the optimisers |
| **Val** | 50 | Held-out validation — used by optimisers to score candidate programs |
| **Test** | 80 | Held-out test — never seen during optimisation; used for final reporting |

The fixed seed makes the split reproducible. The test set must not be used by any optimiser.

In [ ]:
DATASET_PATH = Path("../data/dc_dataset.xlsx")
REQUIRED_COLUMNS = {
    "utterance", "domain", "sub_domain", "ir_utterance_count",
    "alternative_domain", "alternative_sub_domain",
}

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH.resolve()}")

df = pd.read_excel(DATASET_PATH)
missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")
df = df[df["ir_utterance_count"] > 0].reset_index(drop=True)

examples = []
for _, row in df.iterrows():
    ex = dspy.Example(
        utterance=str(row["utterance"]),
        domain=str(row["domain"]),
        sub_domain=str(row["sub_domain"]),
        alternative_domain=str(row["alternative_domain"]) if pd.notna(row.get("alternative_domain")) else None,
        alternative_sub_domain=str(row["alternative_sub_domain"]) if pd.notna(row.get("alternative_sub_domain")) else None,
    ).with_inputs("utterance")
    examples.append(ex)

labels = [e.domain for e in examples]
train_val, test = train_test_split(examples, test_size=80, stratify=labels, random_state=SEED)
labels_tv = [e.domain for e in train_val]
train, val = train_test_split(train_val, test_size=50, stratify=labels_tv, random_state=SEED)

print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")
print(f"\nDomain distribution (test set):")
from collections import Counter
for domain, count in sorted(Counter(e.domain for e in test).items()):
    print(f"  {domain}: {count}")

In [ ]:
def evaluate(program, examples, label=""):
    scores = []
    for ex in examples:
        try:
            pred = program(utterance=ex.utterance)
            scores.append(dc_metric(ex, pred))
        except Exception:
            scores.append(0.0)
    n = len(scores)
    exact       = sum(s == 1.0 for s in scores)
    domain_only = sum(s == 0.5 for s in scores)
    wrong       = sum(s == 0.0 for s in scores)
    weighted    = exact + 0.5 * domain_only
    tag = f"[{label}] " if label else ""
    print(f"{tag}Exact match  (domain + sub_domain): {exact}/{n}  ({exact/n:.1%})")
    print(f"{tag}Domain only  (sub_domain wrong)   : {domain_only}/{n}  ({domain_only/n:.1%})")
    print(f"{tag}Wrong                             : {wrong}/{n}  ({wrong/n:.1%})")
    return {"exact": exact, "exact_pct": exact/n, "domain_only": domain_only, "wrong": wrong,
            "n": n, "weighted": weighted}


In [ ]:
import time

def evaluate_with_stats(program, examples, label="", tracking_lm=None):
    """Evaluate with per-call latency and cost tracking.

    tracking_lm: the dspy.LM whose .history to read for cost.
                 Defaults to the global `lm` (gpt-4.1-nano).
                 Pass the prod_lm when evaluating the production model.
    """
    _lm = tracking_lm if tracking_lm is not None else lm
    scores, latencies, costs = [], [], []

    for ex in examples:
        try:
            hist_before = len(_lm.history)
            t0   = time.perf_counter()
            pred = program(utterance=ex.utterance)
            latency_ms = (time.perf_counter() - t0) * 1000

            scores.append(dc_metric(ex, pred))
            latencies.append(latency_ms)

            # cost is computed by litellm; None on cache hit
            if len(_lm.history) > hist_before:
                costs.append(_lm.history[-1].get("cost") or 0.0)
        except Exception:
            scores.append(0.0)

    n          = len(scores)
    exact      = sum(s == 1.0 for s in scores)
    dom_only   = sum(s == 0.5 for s in scores)
    wrong      = sum(s == 0.0 for s in scores)
    weighted   = exact + 0.5 * dom_only
    avg_lat    = sum(latencies) / len(latencies) if latencies else 0.0
    avg_cost   = sum(costs)    / len(costs)    if costs    else 0.0

    tag = f"[{label}] " if label else ""
    print(f"{tag}Exact match  (domain + sub_domain): {exact}/{n}  ({exact/n:.1%})")
    print(f"{tag}Domain only  (sub_domain wrong)   : {dom_only}/{n}  ({dom_only/n:.1%})")
    print(f"{tag}Wrong                             : {wrong}/{n}  ({wrong/n:.1%})")
    print(f"{tag}Avg latency : {avg_lat:.0f} ms")
    print(f"{tag}Avg cost    : ${avg_cost:.6f} per call  (${avg_cost * 1000:.4f} per 1k calls)")

    return {
        "exact": exact, "domain_only": dom_only, "wrong": wrong, "n": n,
        "weighted": weighted,
        "avg_latency_ms": avg_lat,
        "avg_cost_usd":   avg_cost,
    }


In [ ]:
# ── Smoke test — run this before the baseline to catch API/auth errors early ──
# If this raises, fix the error shown before running the baseline.
try:
    _test = DCModule()(utterance="I need to book an appointment.")
    print("Smoke test passed.")
    print(f"  domain     = {_test.domain}")
    print(f"  sub_domain = {_test.sub_domain}")
except Exception as e:
    print(f"Smoke test FAILED — all baseline scores will be 0.\nError: {e}")


## 4. Benchmark variants

### 4.1 Production reference: full original prompt on gpt-5.4

This is the production reference: `gpt-5.4` with the complete taxonomy, rules, and six original hand-authored examples hard-coded in its instruction text. It is evaluated through the same single-utterance harness as the student.

It differs from the student variants in two ways:
- **Model:** `gpt-5.4` instead of `claude-haiku-4-5`
- **Prompt content:** the complete taxonomy, constraints, and six hard-coded examples

The original prompt's six list-based examples, containing 11 classified utterances in total, are retained verbatim in `PROD_INSTRUCTIONS`. No DSPy `demos` are attached to this reference program.

> **Note:** The evaluator calls the classifier once per utterance, but the production examples stay hard-coded in the instructions exactly as prompt content.

In [ ]:
prod_lm = dspy.LM(model=REFERENCE_MODEL, max_tokens=MAX_TOKENS, cache=False)

with dspy.context(lm=prod_lm):
    prod_module = DCModule(signature=ProdSignature)

print("── Production reference: gpt-5.4 + 6 hard-coded examples ──")
with dspy.context(lm=prod_lm):
    prod_scores = evaluate_with_stats(
        prod_module, test, label="prod", tracking_lm=prod_lm
    )

### 4.2 Student baseline: gpt-4.1-mini, zero-shot

Before optimising anything we establish a baseline: `gpt-4.1-mini` with the production system prompt and **no few-shot examples**.

This gives us two things:
- The raw accuracy of the model against the full `gpt-5.4` production reference
- The **latency and cost per call** at inference time

Latency is measured with `time.perf_counter()` around each program call. Cost is read from `lm.history[-1]["cost"]`, which litellm populates from the API response.

In [ ]:
baseline = DCModule()
print("── Baseline: gpt-4.1-mini, prod prompt, 0 demos ──")
base_scores = evaluate_with_stats(baseline, test, label="baseline")

### 4.3 Bootstrap: auto-selected few-shot demonstrations

**BootstrapFewShotWithRandomSearch** finds the best few-shot demonstrations to inject into the prompt automatically.

**Phase 1 — Build a demo pool:**
Run the teacher (gpt-4o-mini) over all 191 training examples. Collect every prediction that scores exactly right (`dc_metric_strict`). These become the candidate pool.

**Phase 2 — Random search:**
Sample 20 random subsets from the pool (up to 8 bootstrapped demos per subset), score each against the 50-example validation set using the gpt-4.1-mini student, keep the best.

The result — `optimised_bs` — is the program with the best automatically selected demonstrations. **GEPA starts from this program**, so it builds on the Bootstrap foundation.

In [ ]:
mini_lm = dspy.LM(model=TEACHER_MODEL, max_tokens=MAX_TOKENS, cache=False)

print("── BootstrapFewShotWithRandomSearch: 20 candidates ──")
optimiser_bs = dspy.BootstrapFewShotWithRandomSearch(
    metric=dc_metric_strict,
    max_bootstrapped_demos=8,
    max_labeled_demos=4,
    num_candidate_programs=20,
    num_threads=4,
)

# Teacher uses gpt-4o-mini to generate higher-quality traces for the demo pool.
# Binding .lm on the predictor overrides the global LM for that predictor only —
# the student (DCModule()) still uses the global haiku LM during val-set evaluation.
teacher_module = DCModule()
teacher_module.predict.lm = mini_lm

optimised_bs = optimiser_bs.compile(
    student=DCModule(),
    trainset=train,
    valset=val,
    teacher=teacher_module,
)
print("\n── Evaluating on held-out test set ──")
bs_scores = evaluate_with_stats(optimised_bs, test, label="bootstrap")

### 4.4 GEPA: failure-driven instruction evolution

Use GEPA after Bootstrap to test whether failure-driven edits improve the instruction without broadly rewriting its taxonomy detail. Pay particular attention to domain-only errors: they expose the fine-grained sub-domain distinctions that the student still misses.

**GEPA** takes a different approach from Bootstrap: instead of selecting demonstrations, it uses an LLM to *reflect on specific failure cases* and propose targeted patches to the instruction text.

The process:
1. **Evaluate** the current program on a minibatch, collecting failures with feedback text
2. **Reflect** — gpt-4o-mini reads the failure feedback and proposes a revised instruction that addresses those specific errors
3. **Score** the revision on the validation set
4. **Select** using a Pareto strategy, then repeat

The key strength of GEPA: its instruction proposals are *informed by what specifically broke*. If the model keeps confusing `RefillPrescription` with `RenewPrescription`, the metric captures that in feedback text, and the reflection LM patches the instruction to sharpen that distinction — rather than rewriting the entire prompt from scratch.

GEPA starts from `optimised_bs` so it inherits the Bootstrap demos. The reflection LM is `gpt-4o-mini` (used only for instruction writing, not for classification).

In [ ]:
from dspy.teleprompt import GEPA
from dspy.teleprompt.gepa.gepa_utils import ScoreWithFeedback

# Stronger model used only for reflection — not for classification
reflection_lm = dspy.LM(model=TEACHER_MODEL, temperature=1.0, max_tokens=4096, cache=False)

def dc_metric_gepa(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """GEPA metric: same 0/0.5/1.0 scoring, plus textual feedback on failures."""
    score       = dc_metric(gold, pred)
    pred_domain = (pred.domain     or "").strip()
    pred_sub    = (pred.sub_domain or "").strip()
    gold_domain = gold.domain.strip()
    gold_sub    = gold.sub_domain.strip()

    if score == 1.0:
        return ScoreWithFeedback(score=1.0, feedback=f"Correct: {gold_domain}/{gold_sub}.")
    elif score == 0.5:
        return ScoreWithFeedback(
            score=0.5,
            feedback=(
                f"Primary domain correct ({gold_domain}) but wrong sub-domain. "
                f"Predicted '{pred_sub}', expected '{gold_sub}'. "
                f"Review the distinction between these two sub-domains under {gold_domain}."
            ),
        )
    else:
        return ScoreWithFeedback(
            score=0.0,
            feedback=(
                f"Completely wrong. Predicted '{pred_domain}/{pred_sub}', "
                f"expected '{gold_domain}/{gold_sub}'. "
                f"The utterance belongs to a different primary domain entirely."
            ),
        )


print("── GEPA medium: reflective instruction evolution, starting from Bootstrap ──")
optimiser_gepa = GEPA(
    metric=dc_metric_gepa,
    auto="medium",
    reflection_lm=reflection_lm,
    num_threads=4,
)
optimised_gepa = optimiser_gepa.compile(
    student=optimised_bs,  # start from Bootstrap: keeps its few-shot demos
    trainset=train,
    valset=val,
)
print("\n── Evaluating on held-out test set ──")
gepa_scores = evaluate_with_stats(optimised_gepa, test, label="gepa")

## 5. Results

**Scoring:**
- **Exact** = domain + sub_domain both correct (1.0 pt)
- **Weighted** = exact × 1.0 + domain-only × 0.5 (partial credit for correct primary domain)
- **dExact / dWtd** = change vs the haiku baseline

**Note on latency and cost:** All four variants track latency and cost. Optimised programs (Bootstrap, GEPA) have longer prompts due to injected demonstrations or revised instructions, so their per-call cost will be higher than the zero-shot baseline — this is expected and reflected in the output.

In [ ]:
n = len(test)

rows = [
    ("Production (gpt-5.4 + full original prompt)",       prod_scores,  "prod"),
    ("Baseline  (gpt-4.1-mini, 0 demos)",                 base_scores,  "base"),
    ("Bootstrap (gpt-4.1-mini)",                          bs_scores,    "opt"),
    ("GEPA      (gpt-4.1-mini)",                          gepa_scores,  "opt"),
]

base_exact    = base_scores["exact"]
base_weighted = base_scores["exact"] + 0.5 * base_scores["domain_only"]

# ── Accuracy comparison ───────────────────────────────────────────────────────
SEP = "-" * 100
print("ACCURACY COMPARISON")
print(f"{'Variant':<50}  {'Exact':>7}  {'Exact%':>7}  {'Weighted':>9}  {'Wtd%':>6}  {'dExact':>7}  {'dWtd':>6}")
print(SEP)
for name, scores, kind in rows:
    exact    = scores["exact"]
    dom_only = scores["domain_only"]
    weighted = exact + 0.5 * dom_only
    is_ref   = (kind == "prod")
    is_base  = (kind == "base")
    d_exact  = f"{exact - base_exact:+d}"         if not is_ref and not is_base else "--"
    d_wtd    = f"{weighted - base_weighted:+.1f}"  if not is_ref and not is_base else "--"
    print(f"{name:<50}  {exact:>4}/{n}  {exact/n:>6.1%}  {weighted:>9.1f}  {weighted/n:>6.1%}  {d_exact:>7}  {d_wtd:>6}")

# ── Inference stats ───────────────────────────────────────────────────────────
print()
print("INFERENCE STATS  (avg per call over the 80-example test set)")
print(SEP)
for name, scores, _ in rows:
    lat  = scores.get("avg_latency_ms", 0.0)
    cost = scores.get("avg_cost_usd", 0.0)
    print(f"  {name:<50}  latency {lat:>6.0f} ms   cost ${cost:.6f}  (${cost*1000:.4f}/1k calls)")

## 6. Interpreting a fresh run

- **Establish the production reference first.** The `gpt-5.4` full-prompt score is the comparison point for every student variant. This notebook intentionally contains no stored outputs, so a run cannot be mistaken for a current benchmark.

- **Check held-out exact match and weighted score together.** A high partial-credit score can conceal the core failure mode here: selecting the right primary domain but the wrong sub-domain.

- **Inspect the GEPA-compiled prompt.** GEPA edits the instruction based on failure feedback, so review its output before accepting it — confirm it preserved the taxonomy detail rather than simplifying it.

- **Use the uncached timing result.** Caching is disabled for all LMs, so measured latency represents actual API calls. Bootstrap and GEPA will show higher cost than the baseline due to injected demonstrations and/or longer instructions.

- **If the haiku gap remains material, change the system rather than only the optimiser.** Prompt compression, confidence-gated escalation to a stronger model, or fine-tuning are the next levers to test.